In [ ]:
"""
Screening Credit Agentic AI - Synthetic Dataset Generator (v3)
============================================================
Generate 8 tabel relasional untuk training model screening kredit retail
banking (5C: Character, Capacity, Collateral, Condition + Capital/Identity).

Semua tabel terhubung lewat NIK (foreign key), KECUALI retail_customer_profile
yang punya application_id sebagai primary key.

UPDATE v3 (dari v2) - KHUSUS untuk kebutuhan graph analytics (node RM & node
industry), TIDAK menyentuh eligibility_score/label sama sekali:
  - rm_master: nama cabang & wilayah diganti pakai nama ASLI BNI area
    Jabodetabek (~48 cabang, bukan 10 fiktif), dikelompokkan ke 4 Kantor
    Wilayah asli. CATATAN JUJUR: pemetaan cabang -> wilayah ini best-effort
    dari data publik (bukan struktur organisasi internal BNI yang
    terverifikasi) - cukup untuk keperluan demo akademis.
  - rm_master: jumlah RM per cabang sekarang BERVARIASI (3-8), bukan fixed 4.
  - rm_master: kolom baru `kapasitas_max_nasabah` (5-25 per RM, senior
    cenderung lebih tinggi dari junior) - INPUT, bukan hasil hitungan.
  - retail_customer_profile: assignment RM sekarang pakai "slot pool"
    dibatasi kapasitas tiap RM (bukan random bebas per cabang seperti v2).
    Urutannya dibalik dari sebelumnya: RM dipilih dulu (dari slot yang
    tersedia), branch_name customer BARU ikut dari branch RM yang kepilih -
    supaya konsisten & tidak ada RM yang jumlah nasabahnya melebihi
    kapasitas_max_nasabah miliknya.
  - Kolom `region` generik di retail_customer_profile (Region 1-4) TETAP
    ADA & TIDAK diubah/dihapus (independen, by design pilihan user) -
    dashboard/graph disarankan pakai `rm_region` (dari join ke rm_master),
    bukan kolom `region` generik ini.
    
UPDATE v2 (dari v1):
  - Tambah 3 kolom baru di retail_customer_profile: jenis_kredit_diajukan,
    tenor_diajukan_bulan, tujuan_penggunaan_kredit (input pengajuan debitur,
    saling konsisten satu sama lain & dgn loan_requested/industry)
  - Fix DSR calculation: sebelumnya hardcode asumsi tenor 36 bulan & bunga
    flat 12%, sekarang pakai tenor & bunga yang BENERAN diajukan debitur
    (bunga beda per jenis kredit: KUR lebih rendah krn subsidi pemerintah)
  - NOTE: perubahan DSR ini berdampak ke eligibility_score & label di setiap
    baris (bukan random baru, tapi formula yang lebih akurat) - artinya
    master_dataset.csv, master_scored.csv, dan ML model Layer 1 kamu perlu
    di-regenerate/retrain ulang setelah pakai generator versi ini.
  - Fungsi kategorikan_kelayakan() (4-level: Layak/Layak Bersyarat/Perlu
    Review Ulang/Tidak Layak) disertakan di akhir file sebagai UTILITY
    terpisah - dipakai nanti saat membangun master_scored.csv, BUKAN
    bagian dari retail_customer_profile.csv.

Cara pakai di Google Colab:
    1. Copy semua isi file ini ke satu cell
    2. Run
    3. 8 file CSV akan tersimpan di /content/dataset/
       (retail_customer_profile.csv, dukcapil.csv, slik_credit_history.csv,
        dhn.csv, agunan_atr_bpn.csv, laporan_keuangan.csv, bank_account.csv,
        rm_master.csv)

PENTING saat load ulang CSV-nya nanti (termasuk di tahap join):
    Selalu paksa NIK dibaca sebagai teks, JANGAN biarkan pandas nebak tipenya,
    kalau tidak, 16 digit NIK bisa kepotong presisinya jadi angka:
        pd.read_csv("dukcapil.csv", dtype={"NIK": str})

Catatan penting: nilai tanah/bangunan per kelurahan di sini adalah ESTIMASI
SINTETIS yang dibuat plausible per tingkatan wilayah (bukan data appraisal
resmi/real) - cukup untuk keperluan training model & demo, BUKAN untuk
keputusan bisnis nyata.
"""

import random
import numpy as np
import pandas as pd
from datetime import date, timedelta
import os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Generator acak TERPISAH untuk field tambahan supaya penambahan-penambahan
# ini TIDAK menggeser urutan angka acak yang dipakai tabel/kolom lain yang
# sudah ada sebelumnya.
CB_RNG = np.random.default_rng(SEED + 1)         # khusus current_balance
RM_RNG = np.random.default_rng(SEED + 2)         # khusus rm_master & assignment RM
CREDIT_APP_RNG = np.random.default_rng(SEED + 3) # khusus kolom pengajuan kredit

N_CUSTOMERS = 3000          # jumlah nasabah/debitur unik
OUT_DIR = "/content/dataset" if os.path.isdir("/content") else "../data/raw"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================================
# 0. REFERENCE / LOOKUP DATA
# =========================================================================

FIRST_NAMES_M = ["Budi","Agus","Andi","Rizky","Dedi","Hendra","Yusuf","Fajar",
    "Wahyu","Bambang","Eko","Rudi","Slamet","Joko","Hadi","Ahmad","Dimas",
    "Arif","Taufik","Iwan","Gunawan","Sutrisno","Anton","Rian","Doni",
    "Yudi","Fauzi","Irfan","Bayu","Krisna"]
FIRST_NAMES_F = ["Siti","Dewi","Rina","Ani","Wulan","Sri","Yuni","Fitri",
    "Indah","Lestari","Ratna","Maya","Putri","Ika","Novi","Wati","Ayu",
    "Dian","Rita","Nina","Sari","Yanti","Lina","Desi","Tri","Retno",
    "Kartika","Anggi","Melati","Suryani"]
LAST_NAMES = ["Santoso","Wijaya","Kurniawan","Saputra","Setiawan","Pratama",
    "Hidayat","Nugroho","Firmansyah","Susanto","Gunawan","Halim","Wibowo",
    "Permana","Suryadi","Handoko","Kusuma","Rahman","Siregar","Simanjuntak",
    "Tanjung","Lubis","Hutapea","Panjaitan","Situmorang"]

RELIGIONS = ["ISLAM","KRISTEN","KATOLIK","HINDU","BUDDHA","KONGHUCU"]
MARITAL = ["Menikah","Belum Menikah","Cerai Hidup","Cerai Mati"]
EDUCATION = ["SMA/SMK","D3","S1","S2"]
BLOOD_TYPE = ["A","B","AB","O"]

INDUSTRIES = {
    "Perdagangan": ["Distributor Elektronik","Toko Sembako","Grosir Pakaian",
                    "Distributor Bahan Bangunan","Toko Alat Tulis"],
    "Kuliner": ["Restoran","Katering","Warung Makan","Bakery"],
    "Jasa": ["Bengkel","Laundry","Percetakan","Jasa Konstruksi Kecil"],
    "Manufaktur": ["Konveksi","Furniture","Pengolahan Makanan Ringan"],
    "Pertanian": ["Distributor Hasil Tani","Peternakan Ayam"],
    "Transportasi": ["Ekspedisi Kecil","Rental Kendaraan"],
}
INDUSTRY_RISK = {
    "Perdagangan": 0.05, "Kuliner": 0.10, "Jasa": 0.05,
    "Manufaktur": 0.08, "Pertanian": 0.15, "Transportasi": 0.12,
}

PROVINCES_CITIES = {
    "DKI Jakarta": ["Jakarta Selatan","Jakarta Pusat","Jakarta Timur","Jakarta Barat","Jakarta Utara"],
    "Jawa Barat": ["Bekasi","Depok","Bogor","Tangerang Selatan"],
    "Banten": ["Tangerang"],
}
# Kolom "region" GENERIK ini TETAP ADA & independen (by design, keputusan user) -
# dipakai di retail_customer_profile, TIDAK dihubungkan ke rm_region.
REGIONS = ["Region 1","Region 2","Region 3","Region 4"]

# =========================================================================
# BARU v3: cabang ASLI BNI area Jabodetabek, dikelompokkan ke 4 Kantor
# Wilayah asli. CATATAN: pemetaan cabang->wilayah ini best-effort dari data
# publik (nama cabang & wilayah nyata), BUKAN struktur organisasi internal
# BNI yang terverifikasi - representasi geografis masuk akal untuk demo,
# bukan klaim akurasi 100% terhadap struktur BNI sesungguhnya.
# =========================================================================
WILAYAH_BRANCHES = {
    "Kantor Wilayah 10 Jakarta Senayan": [
        "KC Senayan", "KC Gatot Subroto", "KC Jakarta Pusat", "KCP Melawai",
        "KCP Kyai Maja", "KCP Blok M", "KCP Fatmawati", "KCP Pondok Indah",
        "KCP Cilandak", "KCP Pasar Minggu", "KCP Prof. Supomo Tebet", "KCP Tebet Barat",
    ],
    "Kantor Wilayah 12 Jakarta Kota": [
        "KC Harmoni", "KC Gambir", "KC Menteng", "KC Pecenongan",
        "KCP Gajah Mada", "KCP Menteng Raya", "KCP Cikini", "KCP Pluit",
        "KC Kelapa Gading", "KCP Kelapa Gading Bukit", "KCP Sunter",
        "KCP Daan Mogot", "KCP Cengkareng", "KCP Kedoya", "KCP Kebon Jeruk",
        "KCP Puri Indah",
    ],
    "Kantor Wilayah 14 Jakarta BSD": [
        "KC BSD", "KCP Tangerang Kota", "KCP Karawaci", "KCP Cipondoh",
        "KCP Serpong", "KCP Ciputat", "KCP Bintaro", "KC Bogor",
        "KCP Bogor Baru", "KCP Cibinong", "KCP Dramaga", "KCP Depok Margonda",
        "KCP Depok Beji", "KCP Sawangan",
    ],
    "Kantor Wilayah 15 Jakarta Kemayoran": [
        "KC Kramat", "KCP Rawamangun", "KCP Cawang", "KCP Jatinegara",
        "KCP Kramat Jati", "KCP Pulo Gadung", "KC Bekasi", "KCP Bekasi Timur",
        "KCP Cikarang", "KCP Jababeka",
    ],
}
BRANCHES = [b for branches in WILAYAH_BRANCHES.values() for b in branches]
BRANCH_TO_REGION = {b: wilayah for wilayah, branches in WILAYAH_BRANCHES.items() for b in branches}

RM_LEVELS = ["Junior RB", "Senior RB"]

KELURAHAN_LOOKUP = [
    ("DKI Jakarta","Jakarta Selatan","Tebet","Tebet Timur", 28, 5.5),
    ("DKI Jakarta","Jakarta Selatan","Kebayoran Baru","Gunung",45, 6.0),
    ("DKI Jakarta","Jakarta Selatan","Pancoran","Duren Tiga", 30, 5.5),
    ("DKI Jakarta","Jakarta Pusat","Menteng","Menteng", 55, 6.5),
    ("DKI Jakarta","Jakarta Pusat","Cikini","Cikini", 40, 6.0),
    ("DKI Jakarta","Jakarta Timur","Kramat Jati","Kramat Jati", 18, 4.5),
    ("DKI Jakarta","Jakarta Timur","Cakung","Cakung Barat", 12, 4.0),
    ("DKI Jakarta","Jakarta Barat","Kebon Jeruk","Sukabumi Selatan", 22, 5.0),
    ("DKI Jakarta","Jakarta Barat","Cengkareng","Cengkareng Barat", 16, 4.2),
    ("DKI Jakarta","Jakarta Utara","Kelapa Gading","Kelapa Gading Barat", 25, 5.2),
    ("DKI Jakarta","Jakarta Utara","Pluit","Pluit", 27, 5.3),
    ("Jawa Barat","Bekasi","Bekasi Barat","Bintara", 9, 3.8),
    ("Jawa Barat","Bekasi","Bekasi Timur","Margahayu", 8, 3.6),
    ("Jawa Barat","Depok","Beji","Kemiri Muka", 10, 3.8),
    ("Jawa Barat","Depok","Sukmajaya","Mekarjaya", 8.5, 3.6),
    ("Jawa Barat","Bogor","Bogor Tengah","Paledang", 7, 3.4),
    ("Jawa Barat","Tangerang Selatan","Serpong","Rawa Buntu", 12, 4.0),
    ("Banten","Tangerang","Karawaci","Bojong Jaya", 9, 3.6),
    ("Banten","Tangerang","Cipondoh","Poris Plawad", 7.5, 3.4),
]

ASSET_TYPES = ["Tanah","Rumah","Ruko","Gudang"]
CERT_TYPES = ["SHM","HGB"]
LOAN_TYPES = ["KMK","KI","KPR","KKB","KK"]
COLLECT_MAP = {1:"Lancar", 2:"Dalam Perhatian Khusus (DPK)", 3:"Kurang Lancar",
                4:"Diragukan", 5:"Macet"}
OTHER_BANKS = ["Bank Mandiri","Bank BCA","Bank BRI","Bank BNI","Bank CIMB Niaga",
    "Bank Danamon","Bank Permata","Bank OCBC NISP","Bank Panin","BPR Mitra Usaha"]
DHN_REASONS = ["Tunggakan kredit >90 hari di bank lain","Terlibat kasus fraud dokumen",
    "Kredit macet yang belum diselesaikan","Cek/giro kosong berulang",
    "Laporan pihak ketiga terkait sengketa usaha"]

KUR_MIKRO_MAX = 50_000_000
KUR_KECIL_MAX = 500_000_000

TENOR_RANGE = {"KUR": (12, 36), "KMK": (12, 24), "KI": (36, 60)}
INTEREST_RATE = {"KUR": 0.06, "KMK": 0.11, "KI": 0.10}
INDUSTRY_KMK_WEIGHT = {
    "Perdagangan": 0.75, "Kuliner": 0.75, "Jasa": 0.60,
    "Manufaktur": 0.35, "Pertanian": 0.40, "Transportasi": 0.35,
}

TUJUAN_TEMPLATE = {
    ("KMK", "Perdagangan"): "Modal kerja pembelian stok barang dagangan {sub}",
    ("KMK", "Kuliner"): "Modal kerja pembelian bahan baku harian usaha {sub}",
    ("KMK", "Jasa"): "Modal kerja operasional usaha {sub}",
    ("KMK", "Manufaktur"): "Modal kerja pembelian bahan baku produksi {sub}",
    ("KMK", "Pertanian"): "Modal kerja operasional usaha {sub}",
    ("KMK", "Transportasi"): "Modal kerja operasional armada {sub}",
    ("KI", "Perdagangan"): "Perluasan/renovasi tempat usaha {sub}",
    ("KI", "Kuliner"): "Renovasi & penambahan peralatan usaha {sub}",
    ("KI", "Jasa"): "Pembelian peralatan/mesin usaha {sub}",
    ("KI", "Manufaktur"): "Pembelian mesin produksi tambahan usaha {sub}",
    ("KI", "Pertanian"): "Pengembangan lahan/kandang usaha {sub}",
    ("KI", "Transportasi"): "Pembelian armada kendaraan tambahan usaha {sub}",
    ("KUR", "Perdagangan"): "Tambahan modal kerja usaha {sub}",
    ("KUR", "Kuliner"): "Tambahan modal kerja usaha {sub}",
    ("KUR", "Jasa"): "Tambahan modal kerja usaha {sub}",
    ("KUR", "Manufaktur"): "Tambahan modal kerja usaha {sub}",
    ("KUR", "Pertanian"): "Tambahan modal kerja usaha {sub}",
    ("KUR", "Transportasi"): "Tambahan modal kerja usaha {sub}",
}


def generate_credit_application_details(loan_requested, industry, sub_industry, rng):
    kandidat = ["KMK", "KI"]
    if loan_requested <= KUR_KECIL_MAX:
        kandidat.append("KUR")

    if loan_requested <= KUR_MIKRO_MAX:
        weights_map = {"KUR": 0.65, "KMK": 0.25, "KI": 0.10}
    else:
        kmk_w = INDUSTRY_KMK_WEIGHT.get(industry, 0.5)
        if "KUR" in kandidat:
            weights_map = {"KUR": 0.30, "KMK": kmk_w * 0.70, "KI": (1 - kmk_w) * 0.70}
        else:
            weights_map = {"KMK": kmk_w, "KI": 1 - kmk_w}

    weights = np.array([weights_map[k] for k in kandidat])
    weights = weights / weights.sum()
    jenis_kredit = rng.choice(kandidat, p=weights)

    tenor_min, tenor_max = TENOR_RANGE[jenis_kredit]
    tenor_options = list(range(tenor_min, tenor_max + 1, 6))
    tenor = int(rng.choice(tenor_options))

    template = TUJUAN_TEMPLATE.get((jenis_kredit, industry), "Modal kerja usaha {sub}")
    tujuan = template.format(sub=sub_industry)

    return jenis_kredit, tenor, tujuan


def random_date(start_year, end_year):
    start = date(start_year, 1, 1)
    end = date(end_year, 8, 22)
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, delta))

KODE_WILAYAH = {
    "Jakarta Selatan": "317401", "Jakarta Pusat": "317101", "Jakarta Timur": "317501",
    "Jakarta Barat": "317301", "Jakarta Utara": "317201",
    "Bekasi": "327501", "Depok": "327601", "Bogor": "327101",
    "Tangerang Selatan": "367401", "Tangerang": "367101",
}

def gen_nik(kota, tanggal_lahir, gender, idx):
    wilayah = KODE_WILAYAH.get(kota, "310101")
    d = tanggal_lahir.day + (40 if gender == "Perempuan" else 0)
    m, y = tanggal_lahir.month, tanggal_lahir.year % 100
    return f"{wilayah}{d:02d}{m:02d}{y:02d}{idx:04d}"


# =========================================================================
# 1. DUKCAPIL
# =========================================================================
def generate_dukcapil(n):
    rows = []
    for i in range(1, n+1):
        gender = random.choice(["Laki-Laki","Perempuan"])
        fname = random.choice(FIRST_NAMES_M if gender=="Laki-Laki" else FIRST_NAMES_F)
        lname = random.choice(LAST_NAMES)
        nama = f"{fname} {lname}"
        prov = random.choice(list(PROVINCES_CITIES.keys()))
        kota = random.choice(PROVINCES_CITIES[prov])
        tgl_lahir = random_date(1965, 2003)
        nik = gen_nik(kota, tgl_lahir, gender, i)
        rows.append({
            "dukcapil_id": f"DKC{i:06d}",
            "NIK": nik,
            "nama": nama,
            "tempat_lahir": kota,
            "tanggal_lahir": tgl_lahir.isoformat(),
            "jenis_kelamin": gender,
            "golongan_darah": random.choice(BLOOD_TYPE),
            "alamat": f"Jl. {random.choice(LAST_NAMES)} No. {random.randint(1,150)}",
            "rt_rw": f"{random.randint(1,12):03d}/{random.randint(1,10):03d}",
            "kelurahan_desa": random.choice(["Sukamaju","Sukajadi","Cempaka Putih",
                "Kebon Baru","Duren Sawit","Rawa Bunga","Cipete","Bintaro"]),
            "kecamatan": random.choice(["Tebet","Kramat Jati","Cengkareng",
                "Bekasi Timur","Sukmajaya","Serpong"]),
            "kota_kabupaten": kota,
            "provinsi": prov,
            "agama": random.choice(RELIGIONS),
            "status_perkawinan": random.choice(MARITAL),
            "pekerjaan": "Wiraswasta",
            "kewarganegaraan": "WNI",
            "berlaku_hingga": "SEUMUR HIDUP",
        })
    return pd.DataFrame(rows)


# =========================================================================
# 2. AGUNAN / ATR-BPN
# =========================================================================
def generate_agunan(dukcapil_df):
    rows = []
    agunan_lookup = {}
    for i, r in enumerate(dukcapil_df.itertuples(), start=1):
        nik = r.NIK
        prov, kota, kec, kel, harga_tanah, harga_bangunan = random.choice(KELURAHAN_LOOKUP)
        asset_type = random.choices(ASSET_TYPES, weights=[0.25,0.30,0.35,0.10])[0]
        land_area = round(np.random.uniform(60, 400), 1)
        building_area = 0.0 if asset_type == "Tanah" else round(land_area * np.random.uniform(0.5, 1.3), 1)
        htn = round(harga_tanah * np.random.uniform(0.85, 1.15), 2)
        hbg = round(harga_bangunan * np.random.uniform(0.85, 1.15), 2)
        nilai_tanah = round(land_area * htn * 1_000_000)
        nilai_bangunan = round(building_area * hbg * 1_000_000)
        total_value = nilai_tanah + nilai_bangunan
        ownership_match = np.random.choice(["Ya","Tidak"], p=[0.94, 0.06])
        row = {
            "atr_bpn_id": f"ATR{i:06d}",
            "NIK": nik,
            "asset_type": asset_type,
            "certificate_type": random.choice(CERT_TYPES),
            "certificate_number": f"{random.randint(10000,99999)}/{kel}",
            "provinsi": prov, "kota": kota, "kecamatan": kec, "kelurahan": kel,
            "land_area_m2": land_area,
            "building_area_m2": building_area,
            "nilai_tanah_per_m2": int(htn * 1_000_000),
            "nilai_bangunan_per_m2": int(hbg * 1_000_000),
            "nilai_tanah_total": nilai_tanah,
            "nilai_bangunan_total": nilai_bangunan,
            "total_collateral_value": total_value,
            "ownership_match": ownership_match,
        }
        rows.append(row)
        agunan_lookup[nik] = row
    return pd.DataFrame(rows), agunan_lookup


# =========================================================================
# 3. SLIK CREDIT HISTORY
# =========================================================================
def generate_slik(dukcapil_df):
    rows = []
    slik_summary = {}
    rid = 1
    for r in dukcapil_df.itertuples():
        nik = r.NIK
        n_loans = np.random.choice([0,1,2,3], p=[0.15,0.40,0.30,0.15])
        worst = 1
        total_installment = 0
        for _ in range(n_loans):
            plafond = int(np.random.choice([25,50,75,100,150,200,300,500]) * 1_000_000)
            outstanding = int(plafond * np.random.uniform(0.2, 0.95))
            tenor = int(np.random.choice([12,24,36,48,60]))
            installment = int(plafond / tenor * np.random.uniform(1.02,1.15))
            collect = np.random.choice([1,2,3,4,5], p=[0.72,0.14,0.07,0.04,0.03])
            worst = max(worst, collect)
            total_installment += installment
            rows.append({
                "slik_record_id": f"SLK{rid:06d}",
                "NIK": nik,
                "inquiry_date": random_date(2024,2026).isoformat(),
                "bank_name": random.choice(OTHER_BANKS),
                "loan_type": random.choice(LOAN_TYPES),
                "plafond": plafond,
                "outstanding_balance": outstanding,
                "installment_amount": installment,
                "tenor_month": tenor,
                "collectability": int(collect),
                "collectability_label": COLLECT_MAP[collect],
            })
            rid += 1
        slik_summary[nik] = {"worst_collect": worst, "total_installment": total_installment, "n_loans": n_loans}
    return pd.DataFrame(rows), slik_summary


# =========================================================================
# 4. DHN
# =========================================================================
def generate_dhn(dukcapil_df, slik_summary):
    rows = []
    dhn_lookup = {}
    for i, r in enumerate(dukcapil_df.itertuples(), start=1):
        nik = r.NIK
        worst = slik_summary[nik]["worst_collect"]
        p_blacklist = {1:0.01, 2:0.03, 3:0.10, 4:0.25, 5:0.45}[worst]
        status = np.random.choice(["Ya","Tidak"], p=[p_blacklist, 1-p_blacklist])
        reason = random.choice(DHN_REASONS) if status == "Ya" else ""
        row = {
            "dhn_id": f"DHN{i:06d}",
            "NIK": nik,
            "status_dhn": status,
            "alasan": reason,
            "tanggal_input": random_date(2023,2026).isoformat(),
        }
        rows.append(row)
        dhn_lookup[nik] = status
    return pd.DataFrame(rows), dhn_lookup


# =========================================================================
# 5. LAPORAN KEUANGAN
# =========================================================================
def generate_laporan_keuangan(dukcapil_df):
    rows = []
    fin_summary = {}
    rid = 1
    for r in dukcapil_df.itertuples():
        nik = r.NIK
        revenue_2024 = np.random.lognormal(mean=16.8, sigma=0.6)
        growth = np.random.normal(0.12, 0.20)
        revenue_2025 = revenue_2024 * (1 + growth)
        margin = np.clip(np.random.normal(0.11, 0.05), 0.01, 0.35)
        recs = []
        for yr, rev in [(2024, revenue_2024), (2025, revenue_2025)]:
            net_profit = rev * margin * np.random.uniform(0.85,1.15)
            total_asset = rev * np.random.uniform(1.1, 2.0)
            total_liability = total_asset * np.random.uniform(0.2, 0.7)
            op_cf = net_profit * np.random.uniform(0.8, 1.4)
            row = {
                "laporan_id": f"FIN{rid:06d}", "NIK": nik, "year": yr,
                "revenue": int(rev), "net_profit": int(net_profit),
                "total_asset": int(total_asset), "total_liability": int(total_liability),
                "operating_cashflow": int(op_cf),
            }
            rows.append(row); recs.append(row); rid += 1
        fin_summary[nik] = {
            "revenue_growth": growth,
            "latest_revenue": recs[1]["revenue"],
            "latest_net_profit": recs[1]["net_profit"],
            "latest_liability": recs[1]["total_liability"],
        }
    return pd.DataFrame(rows), fin_summary


# =========================================================================
# 6. BANK ACCOUNT
# =========================================================================
def generate_bank_account(dukcapil_df, fin_summary):
    rows = []
    cf_summary = {}
    aid = 1
    for r in dukcapil_df.itertuples():
        nik = r.NIK
        n_acc = np.random.choice([1,2], p=[0.65,0.35])
        monthly_rev = fin_summary[nik]["latest_revenue"] / 12
        best_avg_balance = 0
        for _ in range(n_acc):
            avg_credit = monthly_rev * np.random.uniform(0.6, 1.1)
            avg_debit = avg_credit * np.random.uniform(0.7, 0.98)
            avg_balance = max(avg_credit - avg_debit, 0) * np.random.uniform(40, 100)
            best_avg_balance = max(best_avg_balance, avg_balance)

            account = {
                "account_id": f"ACC{aid:06d}",
                "NIK": nik,
                "account_number": f"{random.randint(1000000000,9999999999):010d}",
                "bank_name": random.choice(["BNI"] + OTHER_BANKS),
                "account_type": random.choice(["Giro","Tabungan"]),
                "account_status": np.random.choice(["Aktif","Dormant"], p=[0.93,0.07]),
                "opened_date": random_date(2015,2025).isoformat(),
                "average_balance_6m": int(avg_balance),
                "average_monthly_credit": int(avg_credit),
                "average_monthly_debit": int(avg_debit),
                "transaction_frequency_monthly": int(np.random.uniform(20,200)),
                "overdraft_count_6m": int(np.random.choice([0,0,0,1,2,3], p=[0.6,0.15,0.1,0.08,0.04,0.03])),
            }

            if account["account_status"] == "Dormant":
                current_balance = avg_balance * CB_RNG.uniform(0.05, 0.25)
            elif account["overdraft_count_6m"] > 0 and CB_RNG.random() < 0.12:
                current_balance = -avg_debit * CB_RNG.uniform(0.02, 0.15)
            else:
                current_balance = avg_balance * CB_RNG.uniform(0.4, 1.8)
            account["current_balance"] = int(current_balance)

            rows.append(account)
            aid += 1
        cf_summary[nik] = {"best_avg_balance": best_avg_balance}
    return pd.DataFrame(rows), cf_summary


# =========================================================================
# 6b. RM MASTER (v3: cabang asli, RM per cabang variatif, kapasitas 5-25)
# =========================================================================
RM_PER_BRANCH_RANGE = (3, 8)       # jumlah RM per cabang, BUKAN fixed 4 lagi
KAPASITAS_JUNIOR_RANGE = (5, 18)   # kapasitas_max_nasabah utk Junior RB
KAPASITAS_SENIOR_RANGE = (10, 25)  # kapasitas_max_nasabah utk Senior RB

def generate_rm_master(n_customers):
    rows = []
    rid = 1
    for branch in BRANCHES:
        n_rm_branch = int(RM_RNG.integers(RM_PER_BRANCH_RANGE[0], RM_PER_BRANCH_RANGE[1] + 1))
        for _ in range(n_rm_branch):
            gender = RM_RNG.choice(["Laki-Laki", "Perempuan"])
            fname = RM_RNG.choice(FIRST_NAMES_M if gender == "Laki-Laki" else FIRST_NAMES_F)
            lname = RM_RNG.choice(LAST_NAMES)
            level = RM_RNG.choice(RM_LEVELS, p=[0.6, 0.4])
            kap_range = KAPASITAS_JUNIOR_RANGE if level == "Junior RB" else KAPASITAS_SENIOR_RANGE
            kapasitas = int(RM_RNG.integers(kap_range[0], kap_range[1] + 1))
            rows.append({
                "rm_id": f"RM{rid:04d}",
                "rm_name": f"{fname} {lname}",
                "branch_name": branch,
                "region": BRANCH_TO_REGION[branch],
                "jabatan": "Relationship Banking Officer",
                "level": level,
                "kapasitas_max_nasabah": kapasitas,
                "join_date": (date(2015, 1, 1) + timedelta(
                    days=int(RM_RNG.integers(0, (date(2025, 12, 31) - date(2015, 1, 1)).days)))).isoformat(),
            })
            rid += 1

    rm_df = pd.DataFrame(rows)

    # Safety check: total kapasitas harus >= N_CUSTOMERS, kalau kurang
    # (jarang terjadi tapi bisa krn randomness) top-up beberapa RM secara acak
    # sampai cukup - tidak ada RM yang "dipaksa" melebihi KAPASITAS_SENIOR_RANGE max.
    total_kapasitas = rm_df["kapasitas_max_nasabah"].sum()
    while total_kapasitas < n_customers:
        idx = RM_RNG.integers(0, len(rm_df))
        if rm_df.loc[idx, "kapasitas_max_nasabah"] < KAPASITAS_SENIOR_RANGE[1]:
            rm_df.loc[idx, "kapasitas_max_nasabah"] += 1
            total_kapasitas += 1

    print(f"  rm_master: {len(rm_df)} RM tersebar di {len(BRANCHES)} cabang, "
          f"total kapasitas {total_kapasitas} (kebutuhan {n_customers})")
    return rm_df


def build_rm_slot_pool(rm_df, n_customers):
    """
    Slot pool: tiap rm_id diulang sebanyak kapasitas_max_nasabah miliknya,
    lalu diacak. N_CUSTOMERS slot pertama dipakai untuk assignment -
    menjamin TIDAK ADA RM yang jumlah nasabahnya melebihi kapasitasnya.
    """
    slots = []
    for row in rm_df.itertuples():
        slots.extend([row.rm_id] * row.kapasitas_max_nasabah)
    RM_RNG.shuffle(slots)
    return slots[:n_customers]


# =========================================================================
# 7. RETAIL CUSTOMER PROFILE (application) + LABEL diterima/ditolak
# =========================================================================
def compute_label_score(worst_collect, dhn_status, growth, net_profit, dsr,
                          collateral_ratio, industry):
    s_character = {1:1.0, 2:0.8, 3:0.5, 4:0.25, 5:0.0}[worst_collect]
    if dhn_status == "Ya":
        s_character = min(s_character, 0.1)
    s_capacity = np.clip(0.5 + growth*1.2, 0, 1) * 0.5 + np.clip(1 - dsr, 0, 1) * 0.5
    s_capacity = np.clip(s_capacity, 0, 1)
    s_collateral = np.clip(collateral_ratio / 1.5, 0, 1)
    s_condition = 1 - INDUSTRY_RISK.get(industry, 0.1) * 4
    s_condition = np.clip(s_condition, 0, 1)

    score = 0.35*s_character + 0.30*s_capacity + 0.20*s_collateral + 0.15*s_condition
    score += np.random.normal(0, 0.05)
    return np.clip(score, 0, 1)

def generate_customer_profile(dukcapil_df, agunan_lookup, slik_summary, dhn_lookup,
                                fin_summary, cf_summary, rm_df, rm_slots):
    rm_id_to_branch = dict(zip(rm_df["rm_id"], rm_df["branch_name"]))
    rows = []
    for i, r in enumerate(dukcapil_df.itertuples(), start=1):
        nik = r.NIK
        prov, kota = r.provinsi, r.kota_kabupaten
        legal_entity = random.choice(["PT","CV","UD"])
        industry = random.choice(list(INDUSTRIES.keys()))
        sub_industry = random.choice(INDUSTRIES[industry])
        business_age = int(np.random.uniform(1, 20))
        employee_count = int(np.random.uniform(2, 80))
        monthly_turnover = fin_summary[nik]["latest_revenue"] / 12

        agunan = agunan_lookup[nik]
        loan_requested = int(np.random.choice([50,75,100,150,200,300,500,750,1000]) * 1_000_000)
        loan_requested = min(loan_requested, 10_000_000_000)
        collateral_ratio = round(agunan["total_collateral_value"] / max(loan_requested,1), 2)
        collateral_size_m2 = round(agunan["land_area_m2"] + agunan["building_area_m2"], 1)

        slik = slik_summary[nik]

        jenis_kredit_diajukan, tenor_diajukan_bulan, tujuan_penggunaan_kredit = \
            generate_credit_application_details(loan_requested, industry, sub_industry, CREDIT_APP_RNG)

        annual_rate = INTEREST_RATE[jenis_kredit_diajukan]
        new_installment = loan_requested / tenor_diajukan_bulan * (1 + annual_rate * tenor_diajukan_bulan / 12)
        dsr = (slik["total_installment"] + new_installment) / max(monthly_turnover, 1)

        score = compute_label_score(
            worst_collect=slik["worst_collect"], dhn_status=dhn_lookup[nik],
            growth=fin_summary[nik]["revenue_growth"], net_profit=fin_summary[nik]["latest_net_profit"],
            dsr=dsr, collateral_ratio=collateral_ratio, industry=industry,
        )
        label = "Diterima" if score >= 0.55 else "Ditolak"

        # --- BARU v3: RM diambil dari slot pool (kapasitas-terjamin),
        # branch_name customer IKUT dari cabang RM yang kepilih (bukan
        # dipilih independen lebih dulu seperti v2) ---
        rm_id = rm_slots[i - 1]
        branch_name = rm_id_to_branch[rm_id]

        row = {
            "application_id": f"APP{2026}{i:05d}",
            "NIK": nik,
            "cif_number": f"CIF{1000000+i}",
            "application_date": random_date(2025,2026).isoformat(),
            "customer_type": "UMKM",
            "company_name": f"{random.choice(['PT','CV','UD'])} {random.choice(LAST_NAMES)} {random.choice(['Jaya','Makmur','Sejahtera','Abadi','Mandiri'])}",
            "legal_entity": legal_entity,
            "owner_name": r.nama,
            "owner_gender": "L" if r.jenis_kelamin=="Laki-Laki" else "P",
            "owner_age": date.today().year - int(r.tanggal_lahir[:4]),
            "owner_marital_status": r.status_perkawinan,
            "owner_education": random.choice(EDUCATION),
            "province": prov, "city": kota,
            "district": r.kecamatan,
            "region": random.choice(REGIONS),  # kolom GENERIK, independen, TIDAK diubah
            "branch_name": branch_name,        # BARU v3: ikut dari cabang RM
            "industry": industry, "sub_industry": sub_industry,
            "business_age_year": business_age,
            "employee_count": employee_count,
            "monthly_turnover_est": int(monthly_turnover),
            "transaction_frequency_monthly": int(np.random.uniform(30,200)),
            "loan_requested": loan_requested,
            "jenis_kredit_diajukan": jenis_kredit_diajukan,
            "tenor_diajukan_bulan": tenor_diajukan_bulan,
            "tujuan_penggunaan_kredit": tujuan_penggunaan_kredit,
            "collateral_type": agunan["asset_type"],
            "collateral_location": f"{agunan['kelurahan']}, {agunan['kota']}",
            "collateral_province": agunan["provinsi"], "collateral_city": agunan["kota"],
            "collateral_size_m2": collateral_size_m2,
            "collateral_market_value": agunan["total_collateral_value"],
            "collateral_liquidation_value": int(agunan["total_collateral_value"] * 0.8),
            "collateral_ratio": collateral_ratio,
            "certificate_type": agunan["certificate_type"],
            "ownership_match": agunan["ownership_match"],
            "estimated_dsr": round(min(dsr,3.0), 2),
            "eligibility_score": round(float(score), 3),
            "label": label,
            "rm_id": rm_id,  # BARU v3: dari slot pool, bukan RM_RNG.choice bebas
        }

        rows.append(row)
    return pd.DataFrame(rows)


# =========================================================================
# UTILITY (dipakai di tahap scoring/master_scored.csv, BUKAN generator ini)
# =========================================================================
def kategorikan_kelayakan(eligibility_score: float) -> str:
    if eligibility_score >= 0.80:
        return "Layak"
    elif eligibility_score >= 0.55:
        return "Layak Bersyarat"
    elif eligibility_score >= 0.40:
        return "Perlu Review Ulang"
    else:
        return "Tidak Layak"


# =========================================================================
# MAIN
# =========================================================================
def main():
    print(f"Generating {N_CUSTOMERS} customers...")
    dukcapil_df = generate_dukcapil(N_CUSTOMERS)
    agunan_df, agunan_lookup = generate_agunan(dukcapil_df)
    slik_df, slik_summary = generate_slik(dukcapil_df)
    dhn_df, dhn_lookup = generate_dhn(dukcapil_df, slik_summary)
    fin_df, fin_summary = generate_laporan_keuangan(dukcapil_df)
    bank_df, cf_summary = generate_bank_account(dukcapil_df, fin_summary)

    rm_df = generate_rm_master(N_CUSTOMERS)
    rm_slots = build_rm_slot_pool(rm_df, N_CUSTOMERS)

    profile_df = generate_customer_profile(dukcapil_df, agunan_lookup, slik_summary,
                                            dhn_lookup, fin_summary, cf_summary, rm_df, rm_slots)

    tables = {
        "retail_customer_profile": profile_df,
        "dukcapil": dukcapil_df,
        "slik_credit_history": slik_df,
        "dhn": dhn_df,
        "agunan_atr_bpn": agunan_df,
        "laporan_keuangan": fin_df,
        "bank_account": bank_df,
        "rm_master": rm_df,
    }
    for name, df in tables.items():
        if "NIK" in df.columns:
            df["NIK"] = df["NIK"].astype(str)
        if "account_number" in df.columns:
            df["account_number"] = df["account_number"].astype(str)
        path = os.path.join(OUT_DIR, f"{name}.csv")
        df.to_csv(path, index=False)
        print(f"  {name:28s} -> {len(df):6d} baris -> {path}")

    print("\nDistribusi label (retail_customer_profile):")
    print(profile_df["label"].value_counts(normalize=True).round(3))
    print("\nDistribusi jumlah nasabah per RM (harus semua <= kapasitas_max_nasabah):")
    nasabah_per_rm = profile_df["rm_id"].value_counts()
    kap_map = rm_df.set_index("rm_id")["kapasitas_max_nasabah"]
    over_capacity = (nasabah_per_rm > kap_map.reindex(nasabah_per_rm.index)).sum()
    print(f"  min={nasabah_per_rm.min()}, max={nasabah_per_rm.max()}, "
          f"median={nasabah_per_rm.median():.0f}, RM melebihi kapasitas={over_capacity} (harus 0)")
    print("\nSelesai. Semua file CSV ada di:", OUT_DIR)
    return tables

if __name__ == "__main__":
    tables = main()


Generating 3000 customers...


  rm_master: 261 RM tersebar di 48 cabang, total kapasitas 3643 (kebutuhan 3000)


  retail_customer_profile      ->   3000 baris -> ../data/raw\retail_customer_profile.csv
  dukcapil                     ->   3000 baris -> ../data/raw\dukcapil.csv
  slik_credit_history          ->   4416 baris -> ../data/raw\slik_credit_history.csv
  dhn                          ->   3000 baris -> ../data/raw\dhn.csv
  agunan_atr_bpn               ->   3000 baris -> ../data/raw\agunan_atr_bpn.csv
  laporan_keuangan             ->   6000 baris -> ../data/raw\laporan_keuangan.csv
  bank_account                 ->   4052 baris -> ../data/raw\bank_account.csv
  rm_master                    ->    261 baris -> ../data/raw\rm_master.csv

Distribusi label (retail_customer_profile):
label
Diterima    0.849
Ditolak     0.151
Name: proportion, dtype: float64

Distribusi jumlah nasabah per RM (harus semua <= kapasitas_max_nasabah):
  min=3, max=24, median=11, RM melebihi kapasitas=0 (harus 0)

Selesai. Semua file CSV ada di: ../data/raw


# Generate Data TestCase

In [ ]:
"""
GENERATE DATA TESTCASE -- Dynamic Routing Paths (Agentic AI Credit Screening)
=============================================================================
Membuat retail_customer_profile_pengajuan_baru.csv (kolom identik dengan
retail_customer_profile.csv) berisi 13 baris pengajuan baru yang masing-masing
dirancang untuk memicu salah satu dari 12 dynamic routing path berikut:

  1.  NIK invalid ATAU pasangan NIK-nama != Dukcapil -> Stop            (2 kasus: 1a & 1b)
  2.  DHN -> Stop
  3.  SLIK Macet -> Stop
  4.  Character bersih -> lanjut Financial
  5.  Loan > 500 juta -> prioritaskan Collateral
  6.  Revenue turun > 30% -> Supporting Data Retrieval
  7.  Overdraft >= 3 -> Cashflow Deep Check
  8.  Dormant account -> Cashflow Investigation
  9.  Nama sertifikat berbeda -> Legal Verification
  10. DSR tinggi -> Credit Recommendation Review
  11. Confidence tinggi -> Fast Track ML
  12. SHAP menunjukkan Financial dominan -> Planner Summary tekankan Financial

retail_customer_profile.csv TIDAK diubah sama sekali. NIK baru yang dipakai di
sini juga ditambahkan sebagai baris baru ke tabel raw lain (dukcapil,
slik_credit_history, dhn, agunan_atr_bpn, laporan_keuangan, bank_account) agar
lookup lintas tabel (join by NIK) tetap konsisten -- KECUALI kasus 1a (NIK
invalid) yang sengaja TIDAK punya baris pendukung sama sekali, karena di
proses aslinya validasi NIK gagal sebelum data lain sempat di-query.

Nomor ID baris baru melanjutkan nomor terakhir yang sudah ada di masing-masing
file (dihitung otomatis dari data existing, BUKAN di-hardcode).

IDEMPOTENT: cell ini aman dijalankan ulang berkali-kali -- baris test case
lama (dikenali dari NIK yang dipakai di CASES) selalu dibuang dulu dari tabel
raw sebelum baris baru ditambahkan, jadi tidak akan ada duplikat NIK meskipun
cell di-run berulang atau setelah generator utama di-run ulang.
"""

import os
from datetime import date

import numpy as np
import pandas as pd

DATA_DIR = "../data/raw" if os.path.isdir("../data/raw") else "data/raw"

INDUSTRY_RISK = {
    "Perdagangan": 0.05, "Kuliner": 0.10, "Jasa": 0.05,
    "Manufaktur": 0.08, "Pertanian": 0.15, "Transportasi": 0.12,
}
INTEREST_RATE = {"KUR": 0.06, "KMK": 0.11, "KI": 0.10}
KOTA_PROVINSI = {
    "Jakarta Selatan": "DKI Jakarta", "Jakarta Pusat": "DKI Jakarta",
    "Jakarta Timur": "DKI Jakarta", "Jakarta Barat": "DKI Jakarta",
    "Jakarta Utara": "DKI Jakarta", "Bekasi": "Jawa Barat", "Depok": "Jawa Barat",
    "Bogor": "Jawa Barat", "Tangerang Selatan": "Jawa Barat", "Tangerang": "Banten",
}
COLLECT_MAP = {1: "Lancar", 2: "Dalam Perhatian Khusus (DPK)", 3: "Kurang Lancar",
               4: "Diragukan", 5: "Macet"}

# =========================================================================
# DEFINISI 13 KASUS UJI
# =========================================================================
CASES = [
    dict(
        case="1a_NIK_INVALID", routing="NIK invalid -> Stop",
        nik="317401050890300", nik_valid=False, has_support=False,
        nama_dukcapil=None, nama_pengajuan="Rudi Saputra",
        gender="L", tgl_lahir=date(1990, 5, 8),
        kota="Jakarta Selatan", kec="Tebet", kel="Tebet Timur",
        industry="Perdagangan", sub_industry="Toko Sembako",
        business_age=6, employees=5, legal_entity="UD",
        loan_requested=100_000_000, jenis_kredit="KMK", tenor=12,
        revenue_2024=300_000_000, revenue_2025=330_000_000, margin=0.10,
        slik_loans=[(1, 50_000_000, 24, "Bank BCA")],
        dhn_status="Tidak", dhn_alasan="",
        asset_type="Tanah", cert_type="SHM", land_area=80, building_area=0,
        harga_tanah_m2=8_000_000, harga_bangunan_m2=0, ownership_match="Ya",
        account_status="Aktif", overdraft_count=0,
        rm_id="RM0001", branch_name="KCP Melawai",
        wilayah="Kantor Wilayah Jakarta Senayan", region="Region 1",
    ),
    dict(
        case="1b_NIK_MISMATCH", routing="NIK-nama != Dukcapil -> Stop",
        nik="3175011205883002", nik_valid=True, has_support=True,
        nama_dukcapil="Dedi Kurniawan", nama_pengajuan="Dedi Firmansyah",
        gender="L", tgl_lahir=date(1988, 5, 12),
        kota="Jakarta Timur", kec="Kramat Jati", kel="Kramat Jati",
        industry="Jasa", sub_industry="Bengkel",
        business_age=8, employees=6, legal_entity="CV",
        loan_requested=150_000_000, jenis_kredit="KMK", tenor=18,
        revenue_2024=400_000_000, revenue_2025=440_000_000, margin=0.12,
        slik_loans=[(1, 50_000_000, 24, "Bank Mandiri")],
        dhn_status="Tidak", dhn_alasan="",
        asset_type="Ruko", cert_type="SHM", land_area=90, building_area=100,
        harga_tanah_m2=18_000_000, harga_bangunan_m2=4_500_000, ownership_match="Ya",
        account_status="Aktif", overdraft_count=0,
        rm_id="RM0021", branch_name="KCP Prof. Supomo Tebet",
        wilayah="Kantor Wilayah Jakarta Senayan", region="Region 1",
    ),
    dict(
        case="2_DHN_HIT", routing="DHN -> Stop",
        nik="3276011503903003", nik_valid=True, has_support=True,
        nama_dukcapil="Hendra Wijaya", nama_pengajuan="Hendra Wijaya",
        gender="L", tgl_lahir=date(1990, 3, 15),
        kota="Depok", kec="Sukmajaya", kel="Mekarjaya",
        industry="Kuliner", sub_industry="Restoran",
        business_age=5, employees=10, legal_entity="CV",
        loan_requested=200_000_000, jenis_kredit="KMK", tenor=24,
        revenue_2024=600_000_000, revenue_2025=660_000_000, margin=0.11,
        slik_loans=[(2, 75_000_000, 36, "Bank Danamon")],
        dhn_status="Ya", dhn_alasan="Tunggakan kredit >90 hari di bank lain",
        asset_type="Rumah", cert_type="SHM", land_area=120, building_area=100,
        harga_tanah_m2=8_500_000, harga_bangunan_m2=3_600_000, ownership_match="Ya",
        account_status="Aktif", overdraft_count=0,
        rm_id="RM0041", branch_name="KCP Tebet Barat",
        wilayah="Kantor Wilayah Jakarta Senayan", region="Region 2",
    ),
    dict(
        case="3_SLIK_MACET", routing="SLIK Macet -> Stop",
        nik="3174012207853004", nik_valid=True, has_support=True,
        nama_dukcapil="Slamet Nugroho", nama_pengajuan="Slamet Nugroho",
        gender="L", tgl_lahir=date(1985, 7, 22),
        kota="Jakarta Selatan", kec="Kebayoran Baru", kel="Gunung",
        industry="Manufaktur", sub_industry="Furniture",
        business_age=10, employees=15, legal_entity="PT",
        loan_requested=250_000_000, jenis_kredit="KI", tenor=48,
        revenue_2024=700_000_000, revenue_2025=750_000_000, margin=0.10,
        slik_loans=[(5, 300_000_000, 36, "Bank Mandiri"), (1, 50_000_000, 12, "Bank BCA")],
        dhn_status="Tidak", dhn_alasan="",
        asset_type="Gudang", cert_type="HGB", land_area=200, building_area=180,
        harga_tanah_m2=6_000_000, harga_bangunan_m2=3_000_000, ownership_match="Ya",
        account_status="Aktif", overdraft_count=0,
        rm_id="RM0061", branch_name="KCP Pecenongan",
        wilayah="Kantor Wilayah Jakarta Kemayoran", region="Region 1",
    ),
    dict(
        case="4_CHARACTER_BERSIH", routing="Character bersih -> lanjut Financial",
        nik="3275014211923005", nik_valid=True, has_support=True,
        nama_dukcapil="Wulan Kusuma", nama_pengajuan="Wulan Kusuma",
        gender="P", tgl_lahir=date(1992, 11, 2),
        kota="Bekasi", kec="Bekasi Timur", kel="Margahayu",
        industry="Perdagangan", sub_industry="Grosir Pakaian",
        business_age=7, employees=8, legal_entity="UD",
        loan_requested=150_000_000, jenis_kredit="KMK", tenor=18,
        revenue_2024=500_000_000, revenue_2025=560_000_000, margin=0.12,
        slik_loans=[(1, 40_000_000, 12, "Bank BRI")],
        dhn_status="Tidak", dhn_alasan="",
        asset_type="Rumah", cert_type="SHM", land_area=100, building_area=90,
        harga_tanah_m2=8_000_000, harga_bangunan_m2=3_600_000, ownership_match="Ya",
        account_status="Aktif", overdraft_count=0,
        rm_id="RM0081", branch_name="KCP Cikini",
        wilayah="Kantor Wilayah Jakarta Kemayoran", region="Region 2",
    ),
    dict(
        case="5_LOAN_BESAR", routing="Loan > 500jt -> prioritaskan Collateral",
        nik="3671011001803006", nik_valid=True, has_support=True,
        nama_dukcapil="Bambang Hidayat", nama_pengajuan="Bambang Hidayat",
        gender="L", tgl_lahir=date(1980, 1, 10),
        kota="Tangerang", kec="Karawaci", kel="Bojong Jaya",
        industry="Manufaktur", sub_industry="Pengolahan Makanan Ringan",
        business_age=15, employees=40, legal_entity="PT",
        loan_requested=750_000_000, jenis_kredit="KI", tenor=60,
        revenue_2024=2_000_000_000, revenue_2025=2_200_000_000, margin=0.11,
        slik_loans=[(1, 100_000_000, 24, "Bank BCA")],
        dhn_status="Tidak", dhn_alasan="",
        asset_type="Gudang", cert_type="SHM", land_area=400, building_area=350,
        harga_tanah_m2=9_000_000, harga_bangunan_m2=3_600_000, ownership_match="Ya",
        account_status="Aktif", overdraft_count=0,
        rm_id="RM0101", branch_name="KCP Sunter",
        wilayah="Kantor Wilayah Jakarta Kemayoran", region="Region 3",
    ),
    dict(
        case="6_REVENUE_TURUN", routing="Revenue turun >30% -> Supporting Data Retrieval",
        nik="3173012506873007", nik_valid=True, has_support=True,
        nama_dukcapil="Fajar Kusuma", nama_pengajuan="Fajar Kusuma",
        gender="L", tgl_lahir=date(1987, 6, 25),
        kota="Jakarta Barat", kec="Kebon Jeruk", kel="Sukabumi Selatan",
        industry="Transportasi", sub_industry="Ekspedisi Kecil",
        business_age=6, employees=12, legal_entity="CV",
        loan_requested=200_000_000, jenis_kredit="KMK", tenor=24,
        revenue_2024=800_000_000, revenue_2025=520_000_000, margin=0.09,
        slik_loans=[(1, 60_000_000, 24, "Bank Permata")],
        dhn_status="Tidak", dhn_alasan="",
        asset_type="Rumah", cert_type="SHM", land_area=110, building_area=95,
        harga_tanah_m2=7_500_000, harga_bangunan_m2=3_400_000, ownership_match="Ya",
        account_status="Aktif", overdraft_count=0,
        rm_id="RM0121", branch_name="KCP Kedoya",
        wilayah="Kantor Wilayah Jakarta BSD", region="Region 3",
    ),
    dict(
        case="7_OVERDRAFT_TINGGI", routing="Overdraft >=3 -> Cashflow Deep Check",
        nik="3276014809913008", nik_valid=True, has_support=True,
        nama_dukcapil="Ika Lestari", nama_pengajuan="Ika Lestari",
        gender="P", tgl_lahir=date(1991, 9, 8),
        kota="Depok", kec="Beji", kel="Kemiri Muka",
        industry="Jasa", sub_industry="Laundry",
        business_age=4, employees=4, legal_entity="UD",
        loan_requested=100_000_000, jenis_kredit="KMK", tenor=12,
        revenue_2024=300_000_000, revenue_2025=330_000_000, margin=0.10,
        slik_loans=[(1, 30_000_000, 12, "Bank OCBC NISP")],
        dhn_status="Tidak", dhn_alasan="",
        asset_type="Rumah", cert_type="SHM", land_area=80, building_area=70,
        harga_tanah_m2=7_000_000, harga_bangunan_m2=3_200_000, ownership_match="Ya",
        account_status="Aktif", overdraft_count=4,
        rm_id="RM0141", branch_name="KCP Tangerang Kota",
        wilayah="Kantor Wilayah Jakarta BSD", region="Region 2",
    ),
    dict(
        case="8_DORMANT_ACCOUNT", routing="Dormant account -> Cashflow Investigation",
        nik="3271011904833009", nik_valid=True, has_support=True,
        nama_dukcapil="Anton Firmansyah", nama_pengajuan="Anton Firmansyah",
        gender="L", tgl_lahir=date(1983, 4, 19),
        kota="Bogor", kec="Bogor Tengah", kel="Paledang",
        industry="Pertanian", sub_industry="Distributor Hasil Tani",
        business_age=9, employees=7, legal_entity="UD",
        loan_requested=120_000_000, jenis_kredit="KMK", tenor=18,
        revenue_2024=350_000_000, revenue_2025=385_000_000, margin=0.10,
        slik_loans=[(1, 35_000_000, 12, "Bank Panin")],
        dhn_status="Tidak", dhn_alasan="",
        asset_type="Tanah", cert_type="SHM", land_area=150, building_area=0,
        harga_tanah_m2=6_500_000, harga_bangunan_m2=0, ownership_match="Ya",
        account_status="Dormant", overdraft_count=0,
        rm_id="RM0161", branch_name="KC BSD",
        wilayah="Kantor Wilayah Jakarta BSD", region="Region 3",
    ),
    dict(
        case="9_SERTIFIKAT_BEDA", routing="Nama sertifikat berbeda -> Legal Verification",
        nik="3172013012793010", nik_valid=True, has_support=True,
        nama_dukcapil="Yusuf Santoso", nama_pengajuan="Yusuf Santoso",
        gender="L", tgl_lahir=date(1979, 12, 30),
        kota="Jakarta Utara", kec="Kelapa Gading", kel="Kelapa Gading Barat",
        industry="Perdagangan", sub_industry="Distributor Elektronik",
        business_age=12, employees=20, legal_entity="PT",
        loan_requested=300_000_000, jenis_kredit="KI", tenor=36,
        revenue_2024=900_000_000, revenue_2025=990_000_000, margin=0.10,
        slik_loans=[(1, 80_000_000, 24, "Bank BRI")],
        dhn_status="Tidak", dhn_alasan="",
        asset_type="Ruko", cert_type="SHM", land_area=150, building_area=180,
        harga_tanah_m2=25_000_000, harga_bangunan_m2=5_200_000, ownership_match="Tidak",
        account_status="Aktif", overdraft_count=0,
        rm_id="RM0181", branch_name="KCP Rawamangun",
        wilayah="Kantor Wilayah 15 (Jakarta Timur)", region="Region 1",
    ),
    dict(
        case="10_DSR_TINGGI", routing="DSR tinggi -> Credit Recommendation Review",
        nik="3674015402903011", nik_valid=True, has_support=True,
        nama_dukcapil="Sri Handoko", nama_pengajuan="Sri Handoko",
        gender="P", tgl_lahir=date(1990, 2, 14),
        kota="Tangerang Selatan", kec="Serpong", kel="Rawa Buntu",
        industry="Kuliner", sub_industry="Katering",
        business_age=3, employees=5, legal_entity="UD",
        loan_requested=300_000_000, jenis_kredit="KI", tenor=36,
        revenue_2024=140_000_000, revenue_2025=150_000_000, margin=0.08,
        slik_loans=[(2, 150_000_000, 12, "Bank Mandiri")],
        dhn_status="Tidak", dhn_alasan="",
        asset_type="Rumah", cert_type="SHM", land_area=90, building_area=80,
        harga_tanah_m2=4_000_000, harga_bangunan_m2=3_600_000, ownership_match="Ya",
        account_status="Aktif", overdraft_count=1,
        rm_id="RM0201", branch_name="KCP Pulo Gadung",
        wilayah="Kantor Wilayah 15 (Jakarta Timur)", region="Region 4",
    ),
    dict(
        case="11_CONFIDENCE_TINGGI", routing="Confidence tinggi -> Fast Track ML",
        nik="3174010508863012", nik_valid=True, has_support=True,
        nama_dukcapil="Taufik Rahman", nama_pengajuan="Taufik Rahman",
        gender="L", tgl_lahir=date(1986, 8, 5),
        kota="Jakarta Selatan", kec="Pancoran", kel="Duren Tiga",
        industry="Jasa", sub_industry="Percetakan",
        business_age=11, employees=9, legal_entity="CV",
        loan_requested=200_000_000, jenis_kredit="KMK", tenor=24,
        revenue_2024=700_000_000, revenue_2025=840_000_000, margin=0.15,
        slik_loans=[(1, 50_000_000, 12, "Bank BCA")],
        dhn_status="Tidak", dhn_alasan="",
        asset_type="Ruko", cert_type="SHM", land_area=130, building_area=150,
        harga_tanah_m2=20_000_000, harga_bangunan_m2=5_000_000, ownership_match="Ya",
        account_status="Aktif", overdraft_count=0,
        rm_id="RM0221", branch_name="KCP Jababeka",
        wilayah="Kantor Wilayah 15 (Jakarta Timur)", region="Region 1",
    ),
    dict(
        case="12_SHAP_FINANCIAL_DOMINAN", routing="SHAP Financial dominan -> Planner Summary tekankan Financial",
        nik="3171016710933013", nik_valid=True, has_support=True,
        nama_dukcapil="Nina Wibowo", nama_pengajuan="Nina Wibowo",
        gender="P", tgl_lahir=date(1993, 10, 27),
        kota="Jakarta Pusat", kec="Menteng", kel="Menteng",
        industry="Pertanian", sub_industry="Peternakan Ayam",
        business_age=5, employees=10, legal_entity="CV",
        loan_requested=250_000_000, jenis_kredit="KI", tenor=36,
        revenue_2024=500_000_000, revenue_2025=800_000_000, margin=0.16,
        slik_loans=[(2, 80_000_000, 24, "Bank CIMB Niaga")],
        dhn_status="Tidak", dhn_alasan="",
        asset_type="Tanah", cert_type="HGB", land_area=140, building_area=0,
        harga_tanah_m2=5_000_000, harga_bangunan_m2=0, ownership_match="Ya",
        account_status="Aktif", overdraft_count=0,
        rm_id="RM0241", branch_name="KCP Dramaga",
        wilayah="Kantor Wilayah 15 (Jakarta Timur)", region="Region 4",
    ),
]

TUJUAN_TEMPLATE = {
    ("KMK", "Perdagangan"): "Modal kerja pembelian stok barang dagangan {sub}",
    ("KMK", "Kuliner"): "Modal kerja pembelian bahan baku harian usaha {sub}",
    ("KMK", "Jasa"): "Modal kerja operasional usaha {sub}",
    ("KMK", "Manufaktur"): "Modal kerja pembelian bahan baku produksi {sub}",
    ("KMK", "Pertanian"): "Modal kerja operasional usaha {sub}",
    ("KMK", "Transportasi"): "Modal kerja operasional armada {sub}",
    ("KI", "Perdagangan"): "Perluasan/renovasi tempat usaha {sub}",
    ("KI", "Kuliner"): "Renovasi & penambahan peralatan usaha {sub}",
    ("KI", "Jasa"): "Pembelian peralatan/mesin usaha {sub}",
    ("KI", "Manufaktur"): "Pembelian mesin produksi tambahan usaha {sub}",
    ("KI", "Pertanian"): "Pengembangan lahan/kandang usaha {sub}",
    ("KI", "Transportasi"): "Pembelian armada kendaraan tambahan usaha {sub}",
}


def compute_score(worst_collect, dhn_status, growth, dsr, collateral_ratio, industry):
    s_character = {1: 1.0, 2: 0.8, 3: 0.5, 4: 0.25, 5: 0.0}[worst_collect]
    if dhn_status == "Ya":
        s_character = min(s_character, 0.1)
    s_capacity = np.clip(0.5 + growth * 1.2, 0, 1) * 0.5 + np.clip(1 - dsr, 0, 1) * 0.5
    s_capacity = np.clip(s_capacity, 0, 1)
    s_collateral = np.clip(collateral_ratio / 1.5, 0, 1)
    s_condition = np.clip(1 - INDUSTRY_RISK.get(industry, 0.1) * 4, 0, 1)
    score = 0.35 * s_character + 0.30 * s_capacity + 0.20 * s_collateral + 0.15 * s_condition
    return float(np.clip(score, 0, 1))


TEST_NIKS = {c["nik"] for c in CASES}

APPEND_TABLES_META = {
    "dukcapil.csv": "dukcapil_id",
    "slik_credit_history.csv": "slik_record_id",
    "dhn.csv": "dhn_id",
    "agunan_atr_bpn.csv": "atr_bpn_id",
    "laporan_keuangan.csv": "laporan_id",
    "bank_account.csv": "account_id",
}


def load_base(fname):
    """Baca tabel raw & buang baris test case lama (kalau ada) supaya idempotent."""
    path = os.path.join(DATA_DIR, fname)
    df = pd.read_csv(path, dtype={"NIK": str})
    return df[~df["NIK"].isin(TEST_NIKS)].reset_index(drop=True)


def max_id_num(df, id_col, prefix):
    if df.empty:
        return 0
    return int(df[id_col].str.replace(prefix, "", regex=False).astype(int).max())


base_tables = {fname: load_base(fname) for fname in APPEND_TABLES_META}

customer_idx = max_id_num(base_tables["dukcapil.csv"], "dukcapil_id", "DKC")
slik_id = max_id_num(base_tables["slik_credit_history.csv"], "slik_record_id", "SLK")
dhn_id = max_id_num(base_tables["dhn.csv"], "dhn_id", "DHN")
atr_id = max_id_num(base_tables["agunan_atr_bpn.csv"], "atr_bpn_id", "ATR")
fin_id = max_id_num(base_tables["laporan_keuangan.csv"], "laporan_id", "FIN")
acc_id = max_id_num(base_tables["bank_account.csv"], "account_id", "ACC")

profile_rows, dukcapil_rows, slik_rows, dhn_rows = [], [], [], []
agunan_rows, fin_rows, bank_rows = [], [], []
today_year = date.today().year

for i, c in enumerate(CASES, start=1):
    customer_idx += 1
    app_id = f"APP2026{customer_idx:05d}"
    cif = f"CIF{1000000 + customer_idx}"
    prov = KOTA_PROVINSI[c["kota"]]

    # --- collateral (self-declared kalau NIK invalid, verified kalau tidak) ---
    nilai_tanah_total = round(c["land_area"] * c["harga_tanah_m2"])
    nilai_bangunan_total = round(c["building_area"] * c["harga_bangunan_m2"])
    total_collateral_value = nilai_tanah_total + nilai_bangunan_total
    collateral_liquidation_value = int(total_collateral_value * 0.8)
    collateral_ratio = round(total_collateral_value / c["loan_requested"], 2)
    collateral_size_m2 = round(c["land_area"] + c["building_area"], 1)

    # --- financial ---
    growth = (c["revenue_2025"] - c["revenue_2024"]) / c["revenue_2024"]
    monthly_turnover = c["revenue_2025"] / 12

    # --- SLIK & DSR ---
    total_installment = 0
    worst_collect = 1
    case_slik_rows = []
    for collect, plafond, tenor_bln, bank in c["slik_loans"]:
        outstanding = int(plafond * 0.6)
        installment = int(plafond / tenor_bln * 1.08)
        total_installment += installment
        worst_collect = max(worst_collect, collect)
        case_slik_rows.append(dict(
            NIK=c["nik"], bank_name=bank, loan_type="KMK", plafond=plafond,
            outstanding_balance=outstanding, installment_amount=installment,
            tenor_month=tenor_bln, collectability=collect,
            collectability_label=COLLECT_MAP[collect],
        ))

    annual_rate = INTEREST_RATE[c["jenis_kredit"]]
    new_installment = c["loan_requested"] / c["tenor"] * (1 + annual_rate * c["tenor"] / 12)
    dsr_raw = (total_installment + new_installment) / monthly_turnover
    estimated_dsr = round(min(dsr_raw, 3.0), 2)

    score = compute_score(worst_collect, c["dhn_status"], growth, dsr_raw,
                           collateral_ratio, c["industry"])
    label = "Diterima" if score >= 0.55 else "Ditolak"

    tujuan = TUJUAN_TEMPLATE.get((c["jenis_kredit"], c["industry"]),
                                  "Modal kerja usaha {sub}").format(sub=c["sub_industry"])

    # --- retail_customer_profile_pengajuan_baru row ---
    profile_rows.append({
        "application_id": app_id, "NIK": c["nik"], "cif_number": cif,
        "application_date": date(2026, 8, 20).isoformat(),
        "customer_type": "UMKM",
        "company_name": f"{c['legal_entity']} {c['nama_pengajuan'].split()[-1]} {'Jaya' if i % 2 else 'Makmur'}",
        "legal_entity": c["legal_entity"],
        "owner_name": c["nama_pengajuan"],
        "owner_gender": "L" if c["gender"] == "L" else "P",
        "owner_age": today_year - c["tgl_lahir"].year,
        "owner_marital_status": "Menikah",
        "owner_education": "S1",
        "province": prov, "city": c["kota"], "district": c["kec"],
        "region": c["region"], "branch_name": c["branch_name"],
        "industry": c["industry"], "sub_industry": c["sub_industry"],
        "business_age_year": c["business_age"], "employee_count": c["employees"],
        "monthly_turnover_est": int(monthly_turnover),
        "transaction_frequency_monthly": 80,
        "loan_requested": c["loan_requested"],
        "jenis_kredit_diajukan": c["jenis_kredit"],
        "tenor_diajukan_bulan": c["tenor"],
        "tujuan_penggunaan_kredit": tujuan,
        "collateral_type": c["asset_type"],
        "collateral_location": f"{c['kel']}, {c['kota']}",
        "collateral_province": prov, "collateral_city": c["kota"],
        "collateral_size_m2": collateral_size_m2,
        "collateral_market_value": total_collateral_value,
        "collateral_liquidation_value": collateral_liquidation_value,
        "collateral_ratio": collateral_ratio,
        "certificate_type": c["cert_type"],
        "ownership_match": c["ownership_match"],
        "estimated_dsr": estimated_dsr,
        "eligibility_score": round(score, 3),
        "label": label,
        "rm_id": c["rm_id"],
        # kolom bantu untuk dokumentasi test case (BUKAN bagian schema asli,
        # dihapus sebelum disimpan ke CSV -- lihat drop di bawah)
        "_case": c["case"], "_routing_path": c["routing"],
    })

    if not c["has_support"]:
        continue  # kasus 1a: NIK invalid, tidak ada baris pendukung di tabel lain

    # --- dukcapil ---
    dukcapil_rows.append({
        "dukcapil_id": f"DKC{customer_idx:06d}", "NIK": c["nik"],
        "nama": c["nama_dukcapil"], "tempat_lahir": c["kota"],
        "tanggal_lahir": c["tgl_lahir"].isoformat(),
        "jenis_kelamin": "Laki-Laki" if c["gender"] == "L" else "Perempuan",
        "golongan_darah": "O",
        "alamat": f"Jl. {c['nama_dukcapil'].split()[-1]} No. {10 + i}",
        "rt_rw": "005/003", "kelurahan_desa": c["kel"], "kecamatan": c["kec"],
        "kota_kabupaten": c["kota"], "provinsi": prov, "agama": "ISLAM",
        "status_perkawinan": "Menikah", "pekerjaan": "Wiraswasta",
        "kewarganegaraan": "WNI", "berlaku_hingga": "SEUMUR HIDUP",
    })

    # --- slik_credit_history ---
    for row in case_slik_rows:
        slik_id += 1
        slik_rows.append({
            "slik_record_id": f"SLK{slik_id:06d}", "NIK": row["NIK"],
            "inquiry_date": date(2026, 6, 1).isoformat(), "bank_name": row["bank_name"],
            "loan_type": row["loan_type"], "plafond": row["plafond"],
            "outstanding_balance": row["outstanding_balance"],
            "installment_amount": row["installment_amount"], "tenor_month": row["tenor_month"],
            "collectability": row["collectability"],
            "collectability_label": row["collectability_label"],
        })

    # --- dhn ---
    dhn_id += 1
    dhn_rows.append({
        "dhn_id": f"DHN{dhn_id:06d}", "NIK": c["nik"], "status_dhn": c["dhn_status"],
        "alasan": c["dhn_alasan"], "tanggal_input": date(2026, 7, 1).isoformat(),
    })

    # --- agunan_atr_bpn ---
    atr_id += 1
    agunan_rows.append({
        "atr_bpn_id": f"ATR{atr_id:06d}", "NIK": c["nik"], "asset_type": c["asset_type"],
        "certificate_type": c["cert_type"],
        "certificate_number": f"{20000 + atr_id}/{c['kel']}",
        "provinsi": prov, "kota": c["kota"], "kecamatan": c["kec"], "kelurahan": c["kel"],
        "land_area_m2": c["land_area"], "building_area_m2": c["building_area"],
        "nilai_tanah_per_m2": c["harga_tanah_m2"], "nilai_bangunan_per_m2": c["harga_bangunan_m2"],
        "nilai_tanah_total": nilai_tanah_total, "nilai_bangunan_total": nilai_bangunan_total,
        "total_collateral_value": total_collateral_value, "ownership_match": c["ownership_match"],
    })

    # --- laporan_keuangan (2024 & 2025) ---
    for yr, rev in [(2024, c["revenue_2024"]), (2025, c["revenue_2025"])]:
        fin_id += 1
        net_profit = int(rev * c["margin"])
        total_asset = int(rev * 1.5)
        total_liability = int(total_asset * 0.4)
        op_cf = int(net_profit * 1.1)
        fin_rows.append({
            "laporan_id": f"FIN{fin_id:06d}", "NIK": c["nik"], "year": yr,
            "revenue": int(rev), "net_profit": net_profit, "total_asset": total_asset,
            "total_liability": total_liability, "operating_cashflow": op_cf,
        })

    # --- bank_account ---
    acc_id += 1
    avg_credit = int(monthly_turnover * 0.85)
    avg_debit = int(avg_credit * 0.85)
    avg_balance = int(max(avg_credit - avg_debit, 0) * 60)
    if c["account_status"] == "Dormant":
        current_balance = int(avg_balance * 0.15)
    else:
        current_balance = int(avg_balance * 1.1)
    bank_rows.append({
        "account_id": f"ACC{acc_id:06d}", "NIK": c["nik"],
        "account_number": f"{7000000000 + acc_id}",
        "bank_name": "BNI", "account_type": "Giro",
        "account_status": c["account_status"],
        "opened_date": date(2021, 1, 15).isoformat(),
        "average_balance_6m": avg_balance, "average_monthly_credit": avg_credit,
        "average_monthly_debit": avg_debit,
        "transaction_frequency_monthly": 80, "overdraft_count_6m": c["overdraft_count"],
        "current_balance": current_balance,
    })

# =========================================================================
# SIMPAN: retail_customer_profile_pengajuan_baru.csv (BARU, retail_customer_profile.csv TIDAK disentuh)
# =========================================================================
profile_new_df = pd.DataFrame(profile_rows)

print("Ringkasan 13 kasus uji & routing path yang dipicu:")
print(profile_new_df[["_case", "_routing_path", "NIK", "owner_name"]].to_string(index=False))

PROFILE_COLUMNS = [
    "application_id", "NIK", "cif_number", "application_date", "customer_type",
    "company_name", "legal_entity", "owner_name", "owner_gender", "owner_age",
    "owner_marital_status", "owner_education", "province", "city", "district",
    "region", "branch_name", "industry", "sub_industry", "business_age_year",
    "employee_count", "monthly_turnover_est", "transaction_frequency_monthly",
    "loan_requested", "jenis_kredit_diajukan", "tenor_diajukan_bulan",
    "tujuan_penggunaan_kredit", "collateral_type", "collateral_location",
    "collateral_province", "collateral_city", "collateral_size_m2",
    "collateral_market_value", "collateral_liquidation_value", "collateral_ratio",
    "certificate_type", "ownership_match", "estimated_dsr", "eligibility_score",
    "label", "rm_id",
]
assert PROFILE_COLUMNS == list(pd.read_csv(os.path.join(DATA_DIR, "retail_customer_profile.csv"), nrows=0).columns)

profile_new_df[PROFILE_COLUMNS].to_csv(
    os.path.join(DATA_DIR, "retail_customer_profile_pengajuan_baru.csv"), index=False
)
print(f"\n-> retail_customer_profile_pengajuan_baru.csv : {len(profile_new_df)} baris baru disimpan.")

# =========================================================================
# APPEND ke tabel raw lain (NIK baru harus konsisten lintas tabel)
# base_tables sudah bersih dari baris test case lama -> aman di-run ulang
# =========================================================================
APPEND_TABLES = {
    "dukcapil.csv": dukcapil_rows,
    "slik_credit_history.csv": slik_rows,
    "dhn.csv": dhn_rows,
    "agunan_atr_bpn.csv": agunan_rows,
    "laporan_keuangan.csv": fin_rows,
    "bank_account.csv": bank_rows,
}
for fname, new_rows in APPEND_TABLES.items():
    path = os.path.join(DATA_DIR, fname)
    base_df = base_tables[fname]
    new_df = pd.DataFrame(new_rows)
    new_df["NIK"] = new_df["NIK"].astype(str)
    combined = pd.concat([base_df, new_df], ignore_index=True)
    combined.to_csv(path, index=False)
    print(f"-> {fname:28s}: +{len(new_df):3d} baris baru (total {len(combined)})")

print("\nSelesai. retail_customer_profile.csv TIDAK diubah.")


In [2]:
pd.set_option('display.max_columns', None)

for name, df in tables.items():
    print(f"\n--- Columns for table: {name} ---")
    display(df.head())



--- Columns for table: retail_customer_profile ---


,application_id,NIK,cif_number,application_date,customer_type,company_name,legal_entity,owner_name,owner_gender,owner_age,owner_marital_status,owner_education,province,city,district,region,branch_name,industry,sub_industry,business_age_year,employee_count,monthly_turnover_est,transaction_frequency_monthly,loan_requested,jenis_kredit_diajukan,tenor_diajukan_bulan,tujuan_penggunaan_kredit,collateral_type,collateral_location,collateral_province,collateral_city,collateral_size_m2,collateral_market_value,collateral_liquidation_value,collateral_ratio,certificate_type,ownership_match,estimated_dsr,eligibility_score,label,rm_id
0,APP202600001,3276010601750001,CIF1000001,2025-07-08,UMKM,UD Santoso Abadi,UD,Budi Panjaitan,L,51,Menikah,S2,Jawa Barat,Depok,Sukmajaya,Region 2,KCP Puri Indah,Manufaktur,Konveksi,14,9,2672137,66,300000000,KI,54,Pembelian mesin produksi tambahan usaha Konveksi,Rumah,"Poris Plawad, Tangerang",Banten,Tangerang,423.4,2328496000,1862796800,7.76,HGB,Ya,3.0,0.804,Diterima,RM0136
1,APP202600002,3172010301920002,CIF1000002,2025-03-23,UMKM,PT Panjaitan Jaya,CV,Andi Hidayat,L,34,Cerai Hidup,S2,DKI Jakarta,Jakarta Utara,Kramat Jati,Region 1,KCP Jatinegara,Transportasi,Rental Kendaraan,1,3,1807917,39,75000000,KUR,24,Tambahan modal kerja usaha Rental Kendaraan,Rumah,"Pluit, Jakarta Utara",DKI Jakarta,Jakarta Utara,174.8,3724038000,2979230400,49.65,HGB,Ya,3.0,0.641,Diterima,RM0191
2,APP202600003,3671010604800003,CIF1000003,2025-08-20,UMKM,CV Hutapea Mandiri,PT,Doni Pratama,L,46,Cerai Hidup,S1,Banten,Tangerang,Bekasi Timur,Region 4,KCP Depok Beji,Manufaktur,Pengolahan Makanan Ringan,17,33,1698736,59,150000000,KUR,24,Tambahan modal kerja usaha Pengolahan Makanan ...,Tanah,"Sukabumi Selatan, Jakarta Barat",DKI Jakarta,Jakarta Barat,67.0,1681700000,1345360000,11.21,HGB,Ya,3.0,0.459,Ditolak,RM0257
3,APP202600004,3173016001890004,CIF1000004,2025-11-02,UMKM,PT Situmorang Mandiri,UD,Nina Firmansyah,P,37,Menikah,S1,DKI Jakarta,Jakarta Barat,Sukmajaya,Region 3,KCP Depok Margonda,Transportasi,Rental Kendaraan,12,79,1135105,98,200000000,KUR,24,Tambahan modal kerja usaha Rental Kendaraan,Ruko,"Tebet Timur, Jakarta Selatan",DKI Jakarta,Jakarta Selatan,200.6,3647200000,2917760000,18.24,HGB,Ya,3.0,0.641,Diterima,RM0248
4,APP202600005,3275011505030005,CIF1000005,2025-10-11,UMKM,UD Tanjung Sejahtera,PT,Sutrisno Nugroho,L,23,Cerai Hidup,D3,Jawa Barat,Bekasi,Kramat Jati,Region 4,KCP Karawaci,Manufaktur,Furniture,11,5,862772,120,750000000,KI,54,Pembelian mesin produksi tambahan usaha Furniture,Ruko,"Kemiri Muka, Depok",Jawa Barat,Depok,316.3,1978268000,1582614400,2.64,HGB,Ya,3.0,0.688,Diterima,RM0151



--- Columns for table: dukcapil ---


,dukcapil_id,NIK,nama,tempat_lahir,tanggal_lahir,jenis_kelamin,golongan_darah,alamat,rt_rw,kelurahan_desa,kecamatan,kota_kabupaten,provinsi,agama,status_perkawinan,pekerjaan,kewarganegaraan,berlaku_hingga
0,DKC000001,3276010601750001,Budi Panjaitan,Depok,1975-01-06,Laki-Laki,B,Jl. Panjaitan No. 27,011/009,Sukajadi,Sukmajaya,Depok,Jawa Barat,HINDU,Menikah,Wiraswasta,WNI,SEUMUR HIDUP
1,DKC000002,3172010301920002,Andi Hidayat,Jakarta Utara,1992-01-03,Laki-Laki,A,Jl. Rahman No. 51,012/009,Cipete,Kramat Jati,Jakarta Utara,DKI Jakarta,HINDU,Cerai Hidup,Wiraswasta,WNI,SEUMUR HIDUP
2,DKC000003,3671010604800003,Doni Pratama,Tangerang,1980-04-06,Laki-Laki,AB,Jl. Setiawan No. 56,006/002,Sukajadi,Bekasi Timur,Tangerang,Banten,ISLAM,Cerai Hidup,Wiraswasta,WNI,SEUMUR HIDUP
3,DKC000004,3173016001890004,Nina Firmansyah,Jakarta Barat,1989-01-20,Perempuan,A,Jl. Wibowo No. 21,009/005,Rawa Bunga,Sukmajaya,Jakarta Barat,DKI Jakarta,KRISTEN,Menikah,Wiraswasta,WNI,SEUMUR HIDUP
4,DKC000005,3275011505030005,Sutrisno Nugroho,Bekasi,2003-05-15,Laki-Laki,B,Jl. Saputra No. 98,005/008,Rawa Bunga,Kramat Jati,Bekasi,Jawa Barat,KATOLIK,Cerai Hidup,Wiraswasta,WNI,SEUMUR HIDUP



--- Columns for table: slik_credit_history ---


...

,laporan_id,NIK,year,revenue,net_profit,total_asset,total_liability,operating_cashflow
0,FIN000001,3276010601750001,2024,29666491,3405584,48148218,27015813,3039455
1,FIN000002,3276010601750001,2025,32065651,3318494,37084511,18787312,4499938
2,FIN000003,3172010301920002,2024,22347316,2907631,27256315,17484128,3572130
3,FIN000004,3172010301920002,2025,21695009,2602253,35323008,9380894,2863796
4,FIN000005,3671010604800003,2024,23914503,5615079,34076107,17322819,7404291



--- Columns for table: bank_account ---


,account_id,NIK,account_number,bank_name,account_type,account_status,opened_date,average_balance_6m,average_monthly_credit,average_monthly_debit,transaction_frequency_monthly,overdraft_count_6m,current_balance
0,ACC000001,3276010601750001,6956650690,Bank BCA,Giro,Aktif,2023-08-14,10342365,2196060,2092204,148,0,13581790
1,ACC000002,3276010601750001,6225587025,BNI,Giro,Aktif,2020-05-20,19800614,2212722,1737140,76,0,9133735
2,ACC000003,3172010301920002,4729122289,Bank BNI,Tabungan,Aktif,2025-03-21,4014256,1124770,1040039,122,0,1718267
3,ACC000004,3671010604800003,4611355005,Bank BCA,Giro,Dormant,2021-10-23,14489012,1394768,1222864,22,1,3156322
4,ACC000005,3173016001890004,8727347064,Bank CIMB Niaga,Giro,Dormant,2025-08-11,1895269,1172430,1132201,157,0,317322



--- Columns for table: rm_master ---


,rm_id,rm_name,branch_name,region,jabatan,level,kapasitas_max_nasabah,join_date
0,RM0001,Doni Hidayat,KCP Melawai,Kantor Wilayah Jakarta Senayan,Relationship Banking Officer,Junior RB,6,2025-08-29
1,RM0002,Wulan Situmorang,KCP Melawai,Kantor Wilayah Jakarta Senayan,Relationship Banking Officer,Junior RB,17,2016-02-10
2,RM0003,Slamet Kusuma,KCP Melawai,Kantor Wilayah Jakarta Senayan,Relationship Banking Officer,Senior RB,17,2025-07-02
3,RM0004,Slamet Gunawan,KCP Melawai,Kantor Wilayah Jakarta Senayan,Relationship Banking Officer,Senior RB,25,2021-06-13
4,RM0005,Putri Santoso,KCP Melawai,Kantor Wilayah Jakarta Senayan,Relationship Banking Officer,Senior RB,11,2024-11-27


In [3]:
import pandas as pd
import os

DATA_DIR = "../data/raw"  # sesuaikan kalau run di lokal, misal "../data/raw_new"

TABLES = [
    "retail_customer_profile", "dukcapil", "slik_credit_history", "dhn",
    "agunan_atr_bpn", "laporan_keuangan", "bank_account", "rm_master",
]

for name in TABLES:
    path = os.path.join(DATA_DIR, f"{name}.csv")
    df = pd.read_csv(path, dtype={"NIK": str})

    print("="*90)
    print(f"TABEL: {name}  |  shape: {df.shape[0]} baris x {df.shape[1]} kolom")
    print("="*90)
    for col in df.columns:
        print(f"  {col:35s} | {str(df[col].dtype):10s} | contoh: {df[col].iloc[0]}")
    print()

TABEL: retail_customer_profile  |  shape: 3000 baris x 41 kolom
  application_id                      | object     | contoh: APP202600001
  NIK                                 | object     | contoh: 3276010601750001
  cif_number                          | object     | contoh: CIF1000001
  application_date                    | object     | contoh: 2025-07-08
  customer_type                       | object     | contoh: UMKM
  company_name                        | object     | contoh: UD Santoso Abadi
  legal_entity                        | object     | contoh: UD
  owner_name                          | object     | contoh: Budi Panjaitan
  owner_gender                        | object     | contoh: L
  owner_age                           | int64      | contoh: 51
  owner_marital_status                | object     | contoh: Menikah
  owner_education                     | object     | contoh: S2
  province                            | object     | contoh: Jawa Barat
  city                  

TABEL: agunan_atr_bpn  |  shape: 3000 baris x 17 kolom
  atr_bpn_id                          | object     | contoh: ATR000001
  NIK                                 | object     | contoh: 3276010601750001
  asset_type                          | object     | contoh: Rumah
  certificate_type                    | object     | contoh: HGB
  certificate_number                  | object     | contoh: 14452/Poris Plawad
  provinsi                            | object     | contoh: Banten
  kota                                | object     | contoh: Tangerang
  kecamatan                           | object     | contoh: Cipondoh
  kelurahan                           | object     | contoh: Poris Plawad
  land_area_m2                        | float64    | contoh: 187.3
  building_area_m2                    | float64    | contoh: 236.1
  nilai_tanah_per_m2                  | int64      | contoh: 8020000
  nilai_bangunan_per_m2               | int64      | contoh: 3500000
  nilai_tanah_total          

In [4]:
from pathlib import Path
import pandas as pd

# naik satu folder dari notebooks ke Capstone Project DA
ROOT = Path.cwd().parent

master_scored = pd.read_csv(ROOT / "data" / "processed" / "master_scored.csv")
master = pd.read_csv(ROOT / "data" / "processed" / "master_dataset.csv")

print(master_scored.shape)
print(master.shape)

(3000, 53)
(3000, 92)


In [5]:
import pandas as pd

master_scored = pd.read_csv(ROOT / "data" / "processed" / "master_scored.csv")
master = pd.read_csv(ROOT / "data" / "processed" / "master_dataset.csv")

print("MASTER")
print([c for c in master.columns if "tanggal" in c.lower() or "date" in c.lower()])

print("SCORED")
print([c for c in master_scored.columns if "tanggal" in c.lower() or "date" in c.lower()])

MASTER


['application_date', 'join_date']
SCORED
[]


In [6]:
print(master_scored["decision"].value_counts(dropna=False))

print([c for c in master_scored.columns
       if "decision" in c.lower()
       or "status" in c.lower()
       or "approval" in c.lower()
       or "recommend" in c.lower()
       or "kategori" in c.lower()])

decision
Layak                 2065
Layak Bersyarat        618
Tidak Layak            224
Perlu Review Ulang      93
Name: count, dtype: int64
['kategori_tenor_diajukan', 'kategori_nominal_diajukan', 'status_dhn', 'decision', 'kategori_kelayakan_ground_truth', 'insight_kategori']


In [7]:
print(master_scored["decision"].value_counts())

# kalau ada kolom lain yang mirip keputusan
print([c for c in master_scored.columns
       if "decision" in c.lower()
       or "status" in c.lower()
       or "approval" in c.lower()
       or "recommend" in c.lower()])

decision
Layak                 2065
Layak Bersyarat        618
Tidak Layak            224
Perlu Review Ulang      93
Name: count, dtype: int64
['status_dhn', 'decision']
